In [1]:
from getpass import getpass

import pandas as pd
from sqlalchemy import URL, create_engine, text

db_url = URL.create(
    "postgresql+psycopg",
    username="analyst",
    password=getpass("Hasło do PostgreSQL: "),
    host="localhost",
    port=5432,
    database="job_market",
)

engine = create_engine(db_url)

with engine.connect() as connection:
    print("Połączono z bazą:", connection.scalar(text("SELECT current_database()")))

Hasło do PostgreSQL:  ········


Połączono z bazą: job_market


In [2]:
import pandas as pd
from sqlalchemy import text


def top_skills(category: str, experience: str) -> pd.DataFrame:
    """Return the top five tracked skills for a category and experience level."""
    query = text("""
        WITH selected_offers AS (
            SELECT job_id
            FROM public.job_offers
            WHERE category = :category
              AND experience = :experience
        )
        SELECT
            s.skill,
            COUNT(*) AS offers_with_skill,
            (SELECT COUNT(*) FROM selected_offers) AS total_offers,
            ROUND(
                100.0 * COUNT(*) / (SELECT COUNT(*) FROM selected_offers),
                1
            ) AS share_pct
        FROM selected_offers AS o
        JOIN public.offer_skills AS s ON s.job_id = o.job_id
        GROUP BY s.skill
        ORDER BY offers_with_skill DESC, s.skill ASC
        LIMIT 5
    """)

    with engine.connect() as connection:
        return pd.read_sql_query(
            query,
            connection,
            params={"category": category, "experience": experience},
        )

In [3]:
top_skills("Data", "Junior")

,skill,offers_with_skill,total_offers,share_pct
0,Python,10,22,45.5
1,SQL,8,22,36.4
2,Excel,7,22,31.8
3,C++,2,22,9.1
4,Linux,2,22,9.1


In [4]:
top_skills("Data", "Mid")

,skill,offers_with_skill,total_offers,share_pct
0,SQL,131,227,57.7
1,Python,99,227,43.6
2,Azure,40,227,17.6
3,Spark,32,227,14.1
4,GCP,27,227,11.9


In [7]:
import json
import requests


category = "Data"
experience = "Junior"

ranking = top_skills(category, experience)
if ranking.empty:
    raise ValueError(f"Brak wyników dla {category} / {experience}.")

top_skill = ranking.iloc[0]

response = requests.post(
    "http://127.0.0.1:11434/api/chat",
    json={
        "model": "qwen3:4b",
        "messages": [{
            "role": "user",
            "content": (
    "W polu zdanie podsumuj poniższe dane po polsku w jednym zdaniu. "
    f"Kategoria: {category}. Poziom: {experience}. "
    f"Najczęściej wymagana umiejętność: {top_skill['skill']}. "
    f"Liczba ofert z tą umiejętnością: {top_skill['offers_with_skill']}. "
    f"Wszystkie oferty w tej grupie: {top_skill['total_offers']}. "
    f"Udział procentowy: {top_skill['share_pct']}%."
),
        }],
        "format": {
            "type": "object",
            "properties": {"zdanie": {"type": "string"}},
            "required": ["zdanie"],
        },
        "think": False,
        "stream": False,
        "options": {
            "num_gpu": 0,
            "num_ctx": 2048,
            "num_predict": 120,
            "temperature": 0,
        },
    },
    timeout=240,
)
response.raise_for_status()
print(json.loads(response.json()["message"]["content"])["zdanie"])

W grupie ofert na poziomie Junior z kategorią Data, 45,5% ofert wymaga umiejętności Python, co odpowiada 10 ofert z 22 wszystkich ofert w tej grupie.


In [7]:
question = "Jakich umiejętności najczęściej wymagają od juniorów?"

import json
import re

import requests

response = requests.post(
    "http://127.0.0.1:11434/api/chat",
    json={
        "model": "qwen3:4b",
        "messages": [{
            "role": "user",
            "content": (
                "Odczytaj z pytania kategorię IT i poziom doświadczenia. "
                "Zapisz kategorię po angielsku, np. Data lub Backend. "
                "Jeśli pytanie nie podaje któregoś filtra, zwróć dla niego null.\n"
                f"Pytanie: {question}"
            ),
        }],
        "format": {
            "type": "object",
            "properties": {
                "category": {"type": ["string", "null"]},
                "experience": {"enum": ["Junior", "Mid", "Senior", "Expert", None]},
            },
            "required": ["category", "experience"],
        },
        "think": False,
        "stream": False,
        "options": {"num_gpu": 0, "num_ctx": 2048, "num_predict": 80, "temperature": 0},
    },
    timeout=240,
)
response.raise_for_status()

filters = json.loads(response.json()["message"]["content"])
# Only apply a seniority filter explicitly mentioned in the question.
level_patterns = {
    "Junior": r"\b(?:junior(?:zy|a|ów|em|om|ach|ski\w*)?|juniork\w*|młodsz\w*)\b",
    "Mid": r"\b(?:mid(?:dle|a|ów|em|om|ach|zi)?|regular(?:a|ów|em|om|ach)?)\b",
    "Senior": r"\b(?:senior(?:zy|a|ów|em|om|ach|ski\w*)?|seniork\w*|starsz\w*)\b",
    "Expert": r"\b(?:expert\w*|eksper(?:t\w*|ci))\b",
}
levels = [level for level, pattern in level_patterns.items() if re.search(pattern, question, re.I)]
filters["experience"] = levels[0] if len(levels) == 1 else None
print(filters)

{'category': 'null', 'experience': 'Junior'}


In [4]:
category = filters.get("category")
experience = filters.get("experience")

if category and not experience:
    print("Dla jakiego poziomu doświadczenia? Wybierz Junior, Mid, Senior lub Expert.")
elif not category or not experience:
    print("Doprecyzuj pytanie: podaj kategorię i poziom doświadczenia.")
else:
    ranking = top_skills(category, experience)
    if ranking.empty:
        print(f"Brak wyników dla kategorii {category} i poziomu {experience}.")
    else:
        print(f"Top skills: {category} / {experience}")
        display(ranking)
        top_skill = ranking.iloc[0]
        prompt = (
            f"Pytanie użytkownika: {question}\n"
            f"Kategoria: {category}. Poziom: {experience}.\n"
            f"Pierwsza umiejętność w rankingu: {top_skill['skill']}.\n"
            f"Wymagana w {top_skill['offers_with_skill']} "
            f"z {top_skill['total_offers']} ofert ({top_skill['share_pct']}%).\n"
            "W polu zdanie podaj po polsku nazwę umiejętności, obie liczby ofert i procent."
        )

        response = requests.post(
            "http://127.0.0.1:11434/api/chat",
            json={
                "model": "qwen3:4b",
                "messages": [{"role": "user", "content": prompt}],
                "format": {
                    "type": "object",
                    "properties": {"zdanie": {"type": "string"}},
                    "required": ["zdanie"],
                },
                "think": False,
                "stream": False,
                "options": {
                    "num_gpu": 0,
                    "num_ctx": 2048,
                    "num_predict": 120,
                    "temperature": 0,
                },
            },
            timeout=240,
        )
        response.raise_for_status()
        print(json.loads(response.json()["message"]["content"])["zdanie"])

Dla jakiego poziomu doświadczenia? Wybierz Junior, Mid, Senior lub Expert.


In [8]:
def salary_summary(category: str, experience: str) -> pd.DataFrame:
    query = text("""
        SELECT
            COUNT(*) AS eligible_offers,
            ROUND(
                (PERCENTILE_CONT(0.5) WITHIN GROUP (
                    ORDER BY salary_mid_pln_monthly
                ))::numeric,
                0
            ) AS median_salary_midpoint_pln
        FROM public.job_offers
        WHERE category = :category
          AND experience = :experience
          AND salary_analysis_eligible IS TRUE
          AND salary_mid_pln_monthly IS NOT NULL
    """)

    with engine.connect() as connection:
        return pd.read_sql_query(
            query,
            connection,
            params={"category": category, "experience": experience},
        )


salary_summary("Data", "Junior")

,eligible_offers,median_salary_midpoint_pln
0,12,12625.0
